# 2 — The Perceptron with `torch.nn`

## The main goal

In notebook 1 you implemented the perceptron update rule by hand. That rule works, but it only works for the perceptron — change the model and you need a new rule.

Modern deep learning uses a single recipe for *every* model, from a linear classifier to a transformer:

1. **Model.** A function $f_\theta(x)$ with learnable parameters $\theta$.
2. **Loss.** A *differentiable* number $L(\theta)$ that measures how wrong the model is on the data.
3. **Gradient descent.** Compute $\nabla_\theta L$ and step in the opposite direction,
   $$
   \theta \;\leftarrow\; \theta \,-\, \eta \, \nabla_\theta L,
   $$
   where $\eta > 0$ is the **learning rate**.

That's the whole recipe. The only things that change between models are the choice of $f_\theta$ and the choice of $L$.

In this notebook we apply the recipe to the same classifier as in notebook 1. The code will look very different, but it's the code you'll see for *every* neural network you meet later: an `nn.Module` that holds the parameters, a `Dataset` + `DataLoader` that feed the data in batches, a loss function, an optimizer, and a training loop built around `forward → loss → backward → step`.

By the end you'll be able to:

- describe what `nn.Module` and `nn.Linear` give you, and why we use them;
- explain how **autograd** computes $\nabla_\theta L$ without you deriving a single partial derivative by hand;
- feed mini-batches through a custom `Dataset` + `DataLoader`;
- **implement** your own `Perceptron` by subclassing `nn.Module`;
- **design** a training loop that iterates the data loader, updates weights with SGD, and reports loss and accuracy per epoch.

The implementation and the training loop are left to *you*. The sections below cover every ingredient you'll need.

## `nn.Module` basics

Every PyTorch model is a class that inherits from `torch.nn.Module`. You only need two methods:

- `__init__`: create the layers and parameters the model will use. Call `super().__init__()` first.
- `forward(x)`: compute the output from the input.

A minimal non-neural example:

```python
class Squarer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x ** 2


f = Squarer()
f(torch.tensor([1.0, 2.0, 3.0]))   # tensor([1., 4., 9.])
```

Two things to remember:

1. **Call `f(x)`, not `f.forward(x)`.** Writing `f(x)` runs `forward` *and* a few extra things PyTorch needs (autograd, train/eval mode, ...). Writing `f.forward(x)` skips them.
2. **Attributes are tracked automatically.** If you write `self.w = nn.Linear(...)`, PyTorch registers it on the module. That's how `model.parameters()` later finds every learnable tensor without you listing them.

In [ ]:
import torch
import torch.nn as nn


class Squarer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x**2


f = Squarer()
f(torch.tensor([1.0, 2.0, 3.0]))  # tensor([1., 4., 9.])

## `nn.Linear` — the affine layer

A linear classifier does one thing:

$$
y = W x + b
$$

PyTorch gives you exactly this as `nn.Linear`:

```python
layer = nn.Linear(in_features, out_features, bias=True)
```

It holds two learnable tensors:

- `layer.weight` — shape `(out_features, in_features)`
- `layer.bias` — shape `(out_features,)`; absent when `bias=False`

For our perceptron, the input is $d$-dimensional and the output is one scalar (the logit), so `nn.Linear(d, 1)` is what we want. Its output is `x @ W.T + b`, which is the $w^\top x + b$ from the lectures.

**Batches.** PyTorch modules expect the first dimension of the input to be the batch. A `(B, d)` input through `nn.Linear(d, 1)` comes out as `(B, 1)` — one logit per row. If you want shape `(B,)` instead, add `.squeeze(-1)` at the end of your `forward`.

In [ ]:
import torch
import torch.nn as nn

layer = nn.Linear(in_features=2, out_features=1, bias=True)
print("weight:", layer.weight.shape, layer.weight)
print("bias  :", layer.bias.shape, layer.bias)

# A batch of 4 points in R^2 goes in, a batch of 4 scalar logits comes out.
x = torch.randn(4, 2)
y = layer(x)
print("x     :", x.shape)
print("y     :", y.shape)  # torch.Size([4, 1])

## Parameters and autograd

Any tensor stored on a `Module` that has `requires_grad=True` is a **parameter** and shows up in `.parameters()`. Those are the only tensors the optimizer will update.

```python
for name, p in layer.named_parameters():
    print(name, p.shape)
# weight torch.Size([1, 2])
# bias   torch.Size([1])
```

When you call `loss.backward()`, each parameter gets a `.grad` tensor: the derivative of the loss with respect to that parameter. The optimizer reads `.grad` and updates the parameter.

You almost never touch `.grad` yourself. But it's worth seeing the full cycle:

1. `forward` runs → PyTorch records every operation in a graph.
2. `loss.backward()` → walks the graph backward, fills `.grad` on every parameter.
3. `optim.step()` → reads `.grad`, updates parameters.
4. `optim.zero_grad()` → clears `.grad`, ready for the next step.

The next cells explain how step 2 actually works.

### How autodiff works

PyTorch computes derivatives for you. It doesn't look up calculus rules, and it doesn't use finite differences. It uses the **chain rule**, one step at a time.

**The idea.** Any function you write is just a chain of simple operations: `+`, `*`, `**`, `sin`, `exp`, `log`, ... PyTorch knows the derivative of each one by hand. To get the derivative of a complicated function, it multiplies the simple derivatives together using the chain rule.

Example: $\ell = (3x + 2)^2$ is three simple steps:

1. multiply $x$ by $3$
2. add $2$
3. square the result

PyTorch knows the derivative of each step. Multiplying them gives $d\ell / dx$.

**Two directions to walk the chain.** The chain rule is a product of derivatives. You can compute that product from either end:

- **Forward mode**: start at the input $x$, carry derivatives forward. Cost: **one pass per input variable**.
- **Reverse mode**: start at the output $\ell$, carry derivatives backward. Cost: **one pass per output variable**.

Both give the same answer. The difference is cost.

**Why neural networks use reverse mode.** The loss is **one number**. The parameters are **many** — easily millions.

- Forward mode: needs one pass per parameter → **millions of passes** to get all gradients.
- Reverse mode: needs one pass per output → **one pass** gives you every gradient at once.

That's it. That's the whole reason deep learning uses reverse-mode AD (the thing people call "backprop"): one scalar loss, many parameters, reverse mode is cheap for that shape.

**How PyTorch records the work.** While `forward` runs, PyTorch writes down every operation into a **graph**. Each new tensor carries a `grad_fn` — a small object that remembers the op and knows how to compute its local derivative. `loss.backward()` walks this graph from the loss back to the leaves, applying the chain rule node by node.

The graph is built fresh on every forward pass (this is called "define-by-run"). That's why ordinary Python — `if`, `for`, function calls — works inside `forward` without any special treatment.

### A worked example

Let's compute $d\ell/dx$ for $\ell = (3x + 2)^2$ at $x = 1$, by hand.

Give the intermediate steps names:

| step                 | value at $x=1$ |
|----------------------|----------------|
| $t_1 = 3\,x$         | $3$            |
| $t_2 = t_1 + 2$      | $5$            |
| $\ell = t_2^{\,2}$   | $25$           |

Now walk **backward** from $\ell$. Start with $\dfrac{\partial \ell}{\partial \ell} = 1$ and multiply by one local derivative at a time:

$$
\frac{\partial \ell}{\partial t_2} \;=\; 2\,t_2 \;=\; 10
$$

$$
\frac{\partial \ell}{\partial t_1} \;=\; \frac{\partial \ell}{\partial t_2} \cdot \frac{\partial t_2}{\partial t_1} \;=\; 10 \cdot 1 \;=\; 10
$$

$$
\frac{\partial \ell}{\partial x} \;=\; \frac{\partial \ell}{\partial t_1} \cdot \frac{\partial t_1}{\partial x} \;=\; 10 \cdot 3 \;=\; 30
$$

Each step uses two things: the **local** derivative of one simple op, and the gradient coming from **above**. Nothing else. PyTorch does exactly this — let's check.

### The same example, forward mode

Reverse mode walks the chain **backward** (from $\ell$ down to $x$). Forward mode walks it **forward** (from $x$ up to $\ell$). Both apply the chain rule, both give the same answer — the only difference is the direction of travel.

In forward mode we carry **two** numbers at every step: the **value**, and its **derivative with respect to** $x$. The seed is $\dfrac{\partial x}{\partial x} = 1$.

| step                | value | derivative w.r.t. $x$              |
|---------------------|-------|-------------------------------------|
| $x$                 | $1$   | $1$                                 |
| $t_1 = 3\,x$        | $3$   | $3 \cdot 1 = 3$                     |
| $t_2 = t_1 + 2$     | $5$   | $3 + 0 = 3$                         |
| $\ell = t_2^{\,2}$  | $25$  | $2\,t_2 \cdot 3 = 30$               |

Same answer, $\dfrac{\partial \ell}{\partial x} = 30$. Every row uses one local derivative and the derivative carried from the previous row. No graph is needed — the derivative is just another number riding alongside the value.

**So why does deep learning use reverse mode?** Because forward mode is seeded from **one specific input** at a time.

- If $\ell$ had two inputs $x_1, x_2$, you'd need **two** forward passes: one seeded with $(\partial x_1 / \partial x_1,\, \partial x_2 / \partial x_1) = (1, 0)$ to get $\partial \ell / \partial x_1$, another seeded with $(0, 1)$ to get $\partial \ell / \partial x_2$.
- Reverse mode is seeded from the **output**: $\partial \ell / \partial \ell = 1$. **One** backward pass gives every partial at once.

Swap "two inputs" for "a million parameters" and it's obvious: forward mode would need a million passes, reverse mode needs one.

(Forward mode isn't useless — it's the right tool in the *opposite* regime, like computing the Jacobian of a function with few inputs and many outputs. For deep-learning-style losses, reverse is the match.)

In [ ]:
import torch

# Compute dl/dx for l = (3x + 2)^2 at x = 1.
x = torch.tensor(1.0, requires_grad=True)  # a leaf tensor, tracked by autograd
l = (3 * x + 2) ** 2  # forward pass: builds the graph

print("l       =", l.item())  # 25.0
print("grad_fn =", l.grad_fn)  # the op that produced l

l.backward()  # walks the graph backward
print("dl/dx   =", x.grad.item())  # 30.0

# By default only the *leaves* (x here) keep their .grad.
# If we want the intermediates too, we ask for them explicitly.
x2 = torch.tensor(1.0, requires_grad=True)
t1 = 3 * x2
t2 = t1 + 2
l2 = t2**2

t1.retain_grad()
t2.retain_grad()

l2.backward()
print("dl/dt1  =", t1.grad.item())  # 10.0
print("dl/dt2  =", t2.grad.item())  # 10.0
print("dl/dx   =", x2.grad.item())  # 30.0

### Why we call `zero_grad()`

One thing about `.grad` that surprises everyone the first time:

> **`.grad` is added to, not replaced.** Every call to `backward()` *adds* to the existing `.grad`. It does not overwrite it.

Why is that the default? Because if a variable shows up in more than one place in the loss, you want to **sum** the two contributions (that's the sum rule of derivatives). Letting `.grad` accumulate does this summing for you automatically.

The catch: if you run `backward()` twice in a row without clearing `.grad` in between, the old gradient is still there, and the new one piles on top. That's a bug.

```python
x = torch.tensor(1.0, requires_grad=True)
l = x**2 + torch.sin(x)
l.backward()
print(x.grad.item())         # 2*x + cos(x) = 2 + cos(1) ≈ 2.5403

# Run backward again without clearing .grad:
l = x**2 + torch.sin(x)      # rebuild the graph
l.backward()
print(x.grad.item())         # ≈ 5.0806 — doubled
```

This is why the training loop starts each step with `optim.zero_grad()`: it resets `.grad` to zero on every parameter, so the next backward pass writes from a clean slate.

## Loss functions

A loss turns `(prediction, target)` into a single number the optimizer can minimize. For binary classification the right default is:

**`nn.BCEWithLogitsLoss()`** — binary cross-entropy, with the sigmoid already inside. It expects:

- **input**: raw **logits** (any real number), shape `(B,)` or `(B, 1)`.
- **target**: floats in $\{0, 1\}$, same shape.

A *logit* is the raw score $z = w^\top x + b \in \mathbb{R}$, before any sigmoid. Its sign is the predicted class, its magnitude is the confidence. `sigmoid(logit)` gives a probability.

**The exact formula.** Start with the **sigmoid** to turn the logit into a probability of class $1$:

$$
p \;=\; \sigma(z) \;=\; \frac{1}{1 + e^{-z}} \;\in\; (0, 1).
$$

**Binary cross-entropy** on that probability, for a target $y \in \{0, 1\}$, is:

$$
\ell(p, y) \;=\; -\bigl[\, y \,\log p \;+\; (1 - y)\,\log(1 - p) \,\bigr].
$$

Intuition: $-\log p$ is $0$ when $p = 1$ and blows up as $p \to 0$. So if $y = 1$, we pay nothing when the model is confident and right, and a lot when it is confident and wrong. Same reasoning for $y = 0$ via the $-\log(1-p)$ term.

Substituting $p = \sigma(z)$ and simplifying gives the form PyTorch actually evaluates directly on the logit:

$$
\ell(z, y) \;=\; (1 - y)\,z \;+\; \log\!\bigl(1 + e^{-z}\bigr).
$$

You can check the two cases by hand:

- $y = 1$: $\ell = \log(1 + e^{-z})$ — small when $z \gg 0$, grows like $-z$ when $z \ll 0$.
- $y = 0$: $\ell = z + \log(1 + e^{-z}) = \log(1 + e^{z})$ — small when $z \ll 0$, grows like $z$ when $z \gg 0$.

For a mini-batch of $B$ samples, the default reduction is the mean:

$$
L \;=\; \frac{1}{B} \sum_{i=1}^{B} \ell(z_i, y_i).
$$

**Why `BCEWithLogitsLoss` and not `BCELoss`?** `BCEWithLogitsLoss` evaluates the closed form above directly, without ever materializing $\sigma(z)$. If you apply `sigmoid(x)` yourself and pass the result to `BCELoss`, the probability can underflow to exactly $0$ or $1$ for large $|z|$, and the following $\log$ returns $-\infty$. Always prefer the `WithLogits` version.

Other losses you'll see later:

- `nn.CrossEntropyLoss` — multi-class classification (softmax + negative log-likelihood).
- `nn.MSELoss` — regression.

One practical note: our labels `Y` are in $\{-1, +1\}$. BCE needs $\{0, 1\}$, so remap once in the training loop.

In [ ]:
# Plot the sigmoid to see its shape:
# - maps any real logit z into (0, 1)
# - sigma(0) = 0.5, symmetric around the origin
# - saturates: sigma(z) -> 1 as z -> +inf, sigma(z) -> 0 as z -> -inf

import torch
import matplotlib.pyplot as plt

z = torch.linspace(-8, 8, 200)
p = torch.sigmoid(z)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z.numpy(), p.numpy(), lw=2, color="C0")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.scatter([0.0], [0.5], color="C3", zorder=5)
ax.annotate(r"$\sigma(0) = 0.5$", xy=(0, 0.5), xytext=(1.0, 0.35),
            arrowprops=dict(arrowstyle="->", color="C3"), color="C3")
ax.set_xlabel(r"logit $z$")
ax.set_ylabel(r"$\sigma(z) = \dfrac{1}{1 + e^{-z}}$")
ax.set_title("Sigmoid squashes any real number into (0, 1)")
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
import torch
import torch.nn as nn

loss_fn = nn.BCEWithLogitsLoss()

logits = torch.tensor([2.0, -1.5, 0.3])  # raw scores (pre-sigmoid)
targets = torch.tensor([1.0, 0.0, 1.0])  # ground truth in {0, 1}
print("loss:", loss_fn(logits, targets).item())

# Sanity checks:
print("confident + correct:", loss_fn(torch.tensor([10.0]), torch.tensor([1.0])).item())
print("confident + wrong  :", loss_fn(torch.tensor([10.0]), torch.tensor([0.0])).item())

## Optimizers and the training step

An optimizer stores a list of parameters and the rule for updating them from gradients. The plain version is stochastic gradient descent:

```python
optim = torch.optim.SGD(model.parameters(), lr=1e-1)
```

- `model.parameters()` is the iterator every `nn.Module` gives you.
- `lr` (learning rate) is how big each update step is.

**The training step.** The same four lines every time:

```python
optim.zero_grad()              # 1. clear .grad on every parameter
logits = model(x_batch)        # 2. forward
loss   = loss_fn(logits, y_batch)
loss.backward()                # 3. fill .grad via autograd
optim.step()                   # 4. update:  p ← p − lr · p.grad
```

Three things worth knowing:

1. **`zero_grad()` is needed** because `.grad` accumulates (see the autograd cells above).
2. **`loss.backward()` only works on a single number.** That's why loss functions average (or sum) over the batch to give a scalar.
3. **`step()` changes the parameters in place.** After `step()`, the graph that connected the loss to the old parameters is no longer valid. The next iteration rebuilds it by calling `forward` again.

## Epochs, mini-batches, shuffling

A training run is organized in two nested loops:

- **Epoch** — one full pass over the training set.
- **Step** — one parameter update, computed on a small subset of the data.

Inside each epoch we break the data into **mini-batches** and run one step per batch. Three choices for the batch size:

- **1 sample per step** (pure SGD): cheap per step, many updates per epoch, noisy gradients.
- **whole dataset per step** (full-batch GD): smooth gradient, but only one update per epoch, and it may not fit in memory.
- **mini-batch** (e.g. 16 or 32): the practical compromise.

We also **shuffle** the data at the start of each epoch. If the model always saw the points in the same order, the updates across epochs become correlated, and on pathological orderings (e.g. "all positives first, then all negatives") the loss can swing back and forth instead of converging.

You *could* do all of this by hand with `torch.randperm` and a Python `for` loop — but the moment you have more than one dataset, or need to preprocess samples, or want parallel loading, the bookkeeping gets annoying. PyTorch's `Dataset` and `DataLoader` are the standard answer.

## `Dataset` and `DataLoader`

PyTorch splits the data pipeline into two pieces.

**`torch.utils.data.Dataset`** — *what a single example looks like*. You subclass it and implement two methods:

- `__len__(self)` — how many examples are in the dataset;
- `__getitem__(self, i)` — return the *i*-th example, usually as a tuple `(x_i, y_i)`.

A minimal dataset that wraps two tensors:

```python
from torch.utils.data import Dataset

class TensorPairDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.Y[i]
```

That's the whole contract — indexing and length. PyTorch does not require anything else.

**`torch.utils.data.DataLoader`** — *how to iterate over batches of examples*. You give it a dataset and it adds batching, shuffling, and (optionally) parallel loading:

```python
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=16, shuffle=True)

for x_batch, y_batch in loader:
    ...   # one training step
```

What the `DataLoader` does each time you iterate it:

1. Asks the dataset for its length.
2. Builds a random permutation of the indices (when `shuffle=True`).
3. Calls `dataset[i]` for each index in the current batch.
4. Stacks the results into tensors of shape `(batch_size, ...)` and yields `(x_batch, y_batch)`.

When the `for` loop ends, one full epoch is done. Wrap it in an outer `for epoch in range(E):` and the whole epoch / shuffle / batching machinery is free.

The last batch may be smaller than `batch_size` if the dataset size is not a multiple (e.g. 64 samples, batch size 16 → four batches of 16; 64 samples, batch size 20 → three batches of 20 and one of 4). Pass `drop_last=True` to `DataLoader` to drop that partial batch, if you prefer fixed-size batches.

(Vocabulary note: a dataset that exposes `__getitem__` and `__len__` is called a **map-style dataset**. There is also `IterableDataset` for streaming scenarios — we don't need it here.)

## Evaluation: `torch.no_grad()` and `.detach()`

When you only want to **use** the model — compute accuracy, plot the decision boundary, evaluate on a test set — you don't need gradients. Wrap the code in:

```python
with torch.no_grad():
    logits = model(X)
    preds = (logits > 0).long()
    acc = (preds == Y_bce.long()).float().mean()
```

Inside `no_grad()`, no graph is built: the code is faster, uses less memory, and doesn't interfere with the next `backward()`.

`.detach()` does the same thing for a single tensor. It returns a copy that no longer participates in the graph. Use it when you want to pull a tensor out of training to plot, save, or pass to numpy:

```python
w = model.w.weight.detach()   # safe to plot / print / save
```

Rule of thumb: **inside training keep the graph, outside training drop it.**

## Creating the dataset

In [ ]:
# Imports

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

In [ ]:
# TODO: implement sample_data(n, w, bias, margin, label, alpha_scale=1.0, perp_scale=3.0)
#   Return a tensor of shape (n, d) whose rows satisfy
#       label * (w^T x + bias) >= margin
#   (Same function as in notebook 1 — feel free to paste it in.)

In [ ]:
# TODO: set the dataset parameters used by the plot below.
#   Need: torch.manual_seed(...), bias (float), margin (float >= 0), w_star (2-D tensor).

In [ ]:
# TODO: generate the dataset.
#   Need: N (int), X (shape (N, 2)), Y (shape (N,) with values in {-1, +1}).
#   Use sample_data twice (once per class), concatenate, and label via X @ w_star + bias.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Split by class for coloring
pos = Y.squeeze() == 1
neg = Y.squeeze() == -1

ax.scatter(
    X[pos, 0],
    X[pos, 1],
    c="tab:blue",
    marker="o",
    s=60,
    label="y = +1",
    edgecolor="k",
)
ax.scatter(
    X[neg, 0],
    X[neg, 1],
    c="tab:orange",
    marker="s",
    s=60,
    label="y = -1",
    edgecolor="k",
)

# Decision boundary w^T x + b = 0 solved for x2:  x2 = -(w0*x1 + b) / w1
xs = np.linspace(X[:, 0].min().item(), X[:, 0].max().item(), 100)
ys = -(w_star[0] * xs + bias) / w_star[1]
ys_plus = -(w_star[0] * xs + bias - margin) / w_star[1]  # w^T x + b = +margin
ys_minus = -(w_star[0] * xs + bias + margin) / w_star[1]  # w^T x + b = -margin
ax.plot(xs, ys, "k--", lw=1.2, label=r"$w^\top x + b = 0$")
ax.plot(
    xs, ys_plus, "k:", lw=1.0, alpha=0.6, label=r"$w^\top x + b = \pm\,\mathrm{margin}$"
)
ax.plot(xs, ys_minus, "k:", lw=1.0, alpha=0.6)

ax.set_xlim(X[:, 0].min().item() - 1, X[:, 0].max().item() + 1)
ax.set_ylim(X[:, 1].min().item() - 1, X[:, 1].max().item() + 1)
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title(f"Random dataset (N={N}, margin={margin}, bias={bias})")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend(loc="best")

plt.tight_layout()
plt.show()

## Your turn

Everything below this point is exercise territory. You'll do three things, using only the machinery introduced above.

**1. Implement `Perceptron` as an `nn.Module`.**

Constructor signature:

```python
class Perceptron(nn.Module):
    def __init__(self, n_input: int = 2, bias: bool = True) -> None:
        ...
```

- `n_input` is the dimensionality of the data.
- `bias` toggles the bias term.
- `forward(x)` should return a tensor of shape `(B,)` — one scalar logit per input row.

Hint: a single `nn.Linear` is all you need.

**2. Wrap the data in a `Dataset` and iterate it with a `DataLoader`.**

- Remap `Y ∈ {-1, +1}` to `Y_bce ∈ {0, 1}` (float) once, up front — `BCEWithLogitsLoss` needs `{0, 1}`.
- Subclass `torch.utils.data.Dataset` with `__len__` and `__getitem__(i) → (x_i, y_i)`, then build an instance `dataset` from `X` and `Y_bce`.
- Instantiate a `DataLoader` with `batch_size=16` and `shuffle=True`.

**3. Design the training loop.**

- Create the model, an SGD optimizer (start with `lr=1e-1`), and a `BCEWithLogitsLoss`.
- For `E` epochs (e.g. `E = 64`), at each epoch:
  1. iterate the `DataLoader` — each step yields a `(x_batch, y_batch)`;
  2. on each batch, run the canonical training block (`zero_grad → forward → loss → backward → step`);
  3. at the end of the epoch, wrap an evaluation pass in `torch.no_grad()` and print loss + accuracy on the full dataset.

All three pieces are implemented in the cells immediately below — treat them as reference after you've given it a shot.

In [ ]:
# TODO: implement the Perceptron class (see "Your turn" above).
#   class Perceptron(nn.Module):
#       def __init__(self, n_input: int = 2, bias: bool = True) -> None: ...
#       def forward(self, x: torch.Tensor) -> torch.Tensor: ...   # shape (B,)

In [ ]:
# TODO: wrap (X, Y) in a torch.utils.data.Dataset.
#   1. Remap Y from {-1, +1} to Y_bce in {0, 1} (float) — BCEWithLogitsLoss needs {0, 1}.
#   2. Subclass Dataset with __len__ and __getitem__(i) -> (x_i, y_i).
#   3. Build an instance `dataset` from X and Y_bce.

In [ ]:
# TODO: build the DataLoader.
#   Need: `loader = DataLoader(dataset, batch_size=16, shuffle=True)`.
#   (Import DataLoader from torch.utils.data.)

In [ ]:
# TODO: instantiate the training ingredients.
#   Need: model (Perceptron), optim (torch.optim.SGD), loss_fn (nn.BCEWithLogitsLoss).
#   Optional: initialize model weights / bias as you like before training.

In [ ]:
# TODO: train the model.
#   For each epoch:
#     - iterate the DataLoader; each step yields (x_batch, y_batch);
#     - on each batch, run the canonical training step
#       (zero_grad -> forward -> loss -> backward -> step);
#     - at the end of the epoch, wrap an evaluation pass in torch.no_grad()
#       and print loss + accuracy on the full dataset.

In [ ]:
# TODO (optional sanity check): count how many points the trained model still
# gets wrong, and print their indices. Should be 0 if training converged.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Split by class for coloring
pos = Y.squeeze() == 1
neg = Y.squeeze() == -1

ax.scatter(
    X[pos, 0],
    X[pos, 1],
    c="tab:blue",
    marker="o",
    s=60,
    label="y = +1",
    edgecolor="k",
)
ax.scatter(
    X[neg, 0],
    X[neg, 1],
    c="tab:orange",
    marker="s",
    s=60,
    label="y = -1",
    edgecolor="k",
)

# True boundary w*^T x + b = 0 solved for x2:  x2 = -(w0*x1 + b) / w1
xs = np.linspace(X[:, 0].min().item(), X[:, 0].max().item(), 100)
ys = -(w_star[0] * xs + bias) / w_star[1]
ys_plus = -(w_star[0] * xs + bias - margin) / w_star[1]  # w*^T x + b = +margin
ys_minus = -(w_star[0] * xs + bias + margin) / w_star[1]  # w*^T x + b = -margin
ax.plot(xs, ys, "k--", lw=1.2, label=r"$w^{*\top} x + b = 0$")
ax.plot(
    xs,
    ys_plus,
    "k:",
    lw=1.0,
    alpha=0.6,
    label=r"$w^{*\top} x + b = \pm\,\mathrm{margin}$",
)
ax.plot(xs, ys_minus, "k:", lw=1.0, alpha=0.6)

# Found weight
with torch.no_grad():
    w_learned = model.w.weight.view(-1)
    xw, yw = w_learned[0].item(), w_learned[1].item()
    bw = model.w.bias.item()

x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
if yw != 0:
    x2 = -(xw * x1 + bw) / yw
    ax.plot(x1, x2, color="red", lw=1.5, label=r"$\hat w^\top x + \hat b = 0$")
else:
    # Vertical line: xw * x1 + bw = 0  =>  x1 = -bw / xw
    ax.axvline(-bw / xw, color="red", lw=1.5, label=r"$\hat w^\top x + \hat b = 0$")

ax.set_xlim(X[:, 0].min().item() - 1, X[:, 0].max().item() + 1)
ax.set_ylim(X[:, 1].min().item() - 1, X[:, 1].max().item() + 1)
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title(f"Random dataset (N={N}, margin={margin}, bias={bias})")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend(loc="best")

plt.tight_layout()
plt.show()

# Appendix — Perceptron ↔ Sigmoid + BCE: a possible connection

## 1. Setup

Inputs $x \in \mathbb{R}^d$, labels $y \in \{-1, +1\}$, linear score

$$
f(x) = w^\top x + b.
$$

The **signed score** is

$$
u \;=\; y \, f(x).
$$

It's positive on correctly classified points, negative on mistakes, and larger means "more confident." Both losses below depend **only on $u$**. That is the whole story.

## 2. Perceptron loss

$$
\ell_{\text{perc}}(u) \;=\; \max(0, -u) \;=\; \begin{cases} 0 & u \ge 0 \\ -u & u < 0 \end{cases}
$$

Zero on correct points, linear on mistakes, non-smooth at $u = 0$.

**Gradient.** When $u < 0$, $\ell_{\text{perc}} = -u = -y\,(w^\top x + b)$, so

$$
\nabla_w \ell_{\text{perc}} = -y\,x, \qquad \partial_b \ell_{\text{perc}} = -y.
$$

When $u \ge 0$, both are $0$. A gradient step is

$$
w \leftarrow w + \eta\, y x, \qquad b \leftarrow b + \eta\, y \qquad \text{if } u \le 0,
$$

and nothing otherwise. With $\eta = 1$, this is exactly Rosenblatt's perceptron rule.

## 3. BCE-with-logits, rewritten in $\{-1, +1\}$

PyTorch's `BCEWithLogitsLoss` uses $t \in \{0, 1\}$ and logit $z \in \mathbb{R}$:

$$
\ell_{\text{BCE}}(t, z) \;=\; -\bigl[\, t \log \sigma(z) + (1 - t)\log(1 - \sigma(z)) \,\bigr], \qquad \sigma(z) = \frac{1}{1 + e^{-z}}.
$$

Using $\log\sigma(z) = -\log(1 + e^{-z})$ and $\log(1 - \sigma(z)) = -\log(1 + e^{z})$:

$$
\ell_{\text{BCE}}(t, z) \;=\; t \log(1 + e^{-z}) \;+\; (1 - t)\log(1 + e^{z}).
$$

Now swap to $\{-1, +1\}$ via $t = (y+1)/2$ (so $t = 1 \iff y = +1$):

$$
\ell(y, z) \;=\; \begin{cases} \log(1 + e^{-z}) & y = +1 \\ \log(1 + e^{+z}) & y = -1 \end{cases} \;=\; \log\!\bigl(1 + e^{-y z}\bigr).
$$

With $z = f(x)$ and $u = y\,f(x)$:

$$
\boxed{\; \ell_{\text{log}}(u) \;=\; \log\!\bigl(1 + e^{-u}\bigr) \;}
$$

So BCE-with-logits **is** the logistic loss, just rewritten with labels in $\{-1, +1\}$.

## 4. Side by side

Both are functions of $u = y f(x)$:

$$
\ell_{\text{perc}}(u) = \max(0, -u), \qquad \ell_{\text{log}}(u) = \log\!\bigl(1 + e^{-u}\bigr).
$$

- Perceptron: zero on correct points, linear on mistakes, non-smooth at $u = 0$.
- Logistic: smooth, strictly decreasing in $u$, strictly positive everywhere — so it keeps pushing for higher confidence even on correct points.

Logistic / BCE is the **smooth surrogate** of the perceptron loss.

## 5. Gradient of the logistic loss

$$
\frac{d\ell_{\text{log}}}{du} \;=\; \frac{-e^{-u}}{1 + e^{-u}} \;=\; -\frac{1}{1 + e^{u}} \;=\; -\sigma(-u).
$$

With $\nabla_w u = y x$ and $\partial_b u = y$:

$$
\nabla_w \ell_{\text{log}} = -\sigma(-u)\, y x, \qquad \partial_b \ell_{\text{log}} = -\sigma(-u)\, y.
$$

A gradient step becomes

$$
w \leftarrow w + \eta\, \sigma(-u)\, y x, \qquad b \leftarrow b + \eta\, \sigma(-u)\, y.
$$

## 6. The soft-perceptron picture

Compare the updates:

$$
\Delta w_{\text{perc}} \;=\; \eta \cdot \mathbf{1}[u \le 0] \cdot y x, \qquad \Delta w_{\text{log}} \;=\; \eta \cdot \sigma(-u) \cdot y x.
$$

Same direction $yx$. Only the weight in front differs:

- **Perceptron** uses the hard indicator $\mathbf{1}[u \le 0]$ — full step on a mistake, nothing on a correct point.
- **Logistic / BCE** uses the smooth gate $\sigma(-u)$:
  - $u \ll 0$ (very wrong) → $\sigma(-u) \approx 1$: near-full perceptron-like step.
  - $u = 0$ (on the boundary) → $\sigma(-u) = 1/2$: half step.
  - $u \gg 0$ (very right) → $\sigma(-u) \approx 0$: almost no step.

So BCE is the perceptron with the hard "is it a mistake?" indicator replaced by its sigmoid relaxation:

$$
\mathbf{1}[u \le 0] \;\longrightarrow\; \sigma(-u).
$$

## 7. Two side notes

**Probabilities.** The perceptron gives only a sign; it has no probability. Logistic / BCE does: $P(y = 1 \mid x) = \sigma(f(x))$. That's why BCE is the right loss when you want calibrated confidence, not just a separating hyperplane.

**Convergence.** Rosenblatt's mistake bound is about the specific update "on mistakes, add $y x$." BCE uses smooth gradient descent on a different loss, so that theorem doesn't transfer directly. Both still respond to the same signed score $u$, which is why they behave similarly on separable data.